# BeamZ Three Layer Matrix Inverse Design


## Purpose

Use the BeamZ backend on the same three exported routing-5 target matrices. This notebook copies targets into `example/targets`, builds the same geometry template, plots BeamZ native permittivity, and prepares layer-by-layer inverse-design calls. BeamZ runs locally for FDTD and adjoint overlap fields while using a JAX implementation of the Lumix/Tidy3D conic filter and smoothed projection for GPU-capable backend parity.


## Imports and Repository Root

Run from the repository environment, for example `uv run --extra tidy3d --extra beamz jupyter lab example/beamz_3layer_inverse_design.ipynb`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'lumix').exists():
            return candidate
    raise RuntimeError('Could not find the LumixTidy3D repository root.')


def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f'{type(value).__name__} is not JSON serializable')


PROJECT_ROOT = find_repo_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

EXAMPLE_DIR = PROJECT_ROOT / 'example'
OUTPUT_BASE = EXAMPLE_DIR / 'outputs'
TARGET_COPY_DIR = EXAMPLE_DIR / 'targets' / 'routing_5_strict'
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
TARGET_COPY_DIR.mkdir(parents=True, exist_ok=True)

from lumix.inverse_design.autograd_runner import MatrixInverseDesignRunConfig, MatrixInverseDesignRunner

PROJECT_ROOT


## Copy Target Matrices Into `example/targets`


In [ ]:
EXPECTED_LAYER_NAMES = ('subunitary_0', 'subunitary_1', 'subunitary_2')
ORIGINAL_TARGETS_PATH = (
    PROJECT_ROOT
    / 'experiments/cases/mnist_pca16/inverse_design_feasible_r4/runs'
    / 'goal93_strict_feasible_review_candidate/routing_5_strict'
    / 'inverse_design_targets/targets.npz'
)
LOCAL_TARGETS_PATH = TARGET_COPY_DIR / 'targets.npz'
FORCE_REFRESH_TARGET_COPY = False


def copy_target_matrices_to_example(*, force: bool = False) -> tuple[Path, dict]:
    if force or not LOCAL_TARGETS_PATH.exists():
        if not ORIGINAL_TARGETS_PATH.exists():
            raise FileNotFoundError(
                f'Missing both local target copy and source target file:\n'
                f'local={LOCAL_TARGETS_PATH}\nsource={ORIGINAL_TARGETS_PATH}'
            )
        shutil.copy2(ORIGINAL_TARGETS_PATH, LOCAL_TARGETS_PATH)

    targets_npz = np.load(LOCAL_TARGETS_PATH)
    missing = [name for name in EXPECTED_LAYER_NAMES if name not in targets_npz.files]
    if missing:
        raise KeyError(f'Local target file is missing layers: {missing}. Available: {targets_npz.files}')

    manifest = {
        'target_set': 'routing_5_strict',
        'source_targets': str(ORIGINAL_TARGETS_PATH),
        'local_targets': str(LOCAL_TARGETS_PATH),
        'layer_order': list(EXPECTED_LAYER_NAMES),
        'layers': {},
    }
    for name in EXPECTED_LAYER_NAMES:
        matrix = np.asarray(targets_npz[name], dtype=np.complex128)
        sidecar = TARGET_COPY_DIR / f'layer_{name}.npy'
        if force or not sidecar.exists():
            np.save(sidecar, matrix)
        singular_values = np.linalg.svd(matrix, compute_uv=False)
        manifest['layers'][name] = {
            'path': str(sidecar),
            'shape': list(matrix.shape),
            'fro_norm': float(np.linalg.norm(matrix)),
            'max_abs': float(np.abs(matrix).max()),
            'mean_abs': float(np.abs(matrix).mean()),
            'max_singular_value': float(singular_values[0]),
            'equivalent_min_loss_db': float(-20.0 * np.log10(max(float(singular_values[0]), 1e-30))),
        }

    manifest_path = TARGET_COPY_DIR / 'target_manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2, default=json_default) + '\n')
    manifest['manifest_path'] = str(manifest_path)
    return LOCAL_TARGETS_PATH, manifest


TARGETS_PATH, target_manifest = copy_target_matrices_to_example(force=FORCE_REFRESH_TARGET_COPY)
target_manifest


## Load Three-Layer Targets


In [ ]:
targets = np.load(TARGETS_PATH)
LAYER_NAMES = EXPECTED_LAYER_NAMES

summary = {}
for name in LAYER_NAMES:
    matrix = np.asarray(targets[name], dtype=np.complex128)
    singular_values = np.linalg.svd(matrix, compute_uv=False)
    summary[name] = {
        'shape': matrix.shape,
        'fro_norm': float(np.linalg.norm(matrix)),
        'max_abs': float(np.abs(matrix).max()),
        'mean_abs': float(np.abs(matrix).mean()),
        'max_singular_value': float(singular_values[0]),
        'equivalent_min_loss_db': float(-20.0 * np.log10(max(float(singular_values[0]), 1e-30))),
    }
summary


## Save Target Matrix Heatmaps


In [ ]:
TARGET_PLOT_DIR = OUTPUT_BASE / 'shared' / 'target_matrices'
TARGET_PLOT_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
for col, name in enumerate(LAYER_NAMES):
    matrix = np.asarray(targets[name], dtype=np.complex128)
    amp = np.abs(matrix)
    phase = np.angle(matrix)

    im0 = axes[0, col].imshow(amp, cmap='plasma', interpolation='nearest')
    axes[0, col].set_title(f'{name} |S|')
    axes[0, col].set_xlabel('input port')
    axes[0, col].set_ylabel('output port')
    fig.colorbar(im0, ax=axes[0, col], fraction=0.046, pad=0.04)

    im1 = axes[1, col].imshow(phase, cmap='twilight', vmin=-np.pi, vmax=np.pi, interpolation='nearest')
    axes[1, col].set_title(f'{name} phase')
    axes[1, col].set_xlabel('input port')
    axes[1, col].set_ylabel('output port')
    fig.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)

summary_plot_path = TARGET_PLOT_DIR / 'routing_5_strict_three_layer_targets_abs_phase.png'
fig.savefig(summary_plot_path, dpi=180)
plt.show()

individual_paths = {}
for name in LAYER_NAMES:
    matrix = np.asarray(targets[name], dtype=np.complex128)
    np.save(TARGET_PLOT_DIR / f'{name}_target_matrix.npy', matrix)
    for label, data, cmap, kwargs in (
        ('abs', np.abs(matrix), 'plasma', {}),
        ('phase', np.angle(matrix), 'twilight', {'vmin': -np.pi, 'vmax': np.pi}),
    ):
        fig, ax = plt.subplots(figsize=(5.5, 4.8), constrained_layout=True)
        im = ax.imshow(data, cmap=cmap, interpolation='nearest', **kwargs)
        ax.set_title(f'{name} {label}')
        ax.set_xlabel('input port')
        ax.set_ylabel('output port')
        fig.colorbar(im, ax=ax)
        path = TARGET_PLOT_DIR / f'{name}_target_{label}.png'
        fig.savefig(path, dpi=180)
        plt.close(fig)
        individual_paths[f'{name}_{label}'] = str(path)

{'summary_plot': str(summary_plot_path), 'individual_plots': individual_paths}


## Category Archive Helpers


In [ ]:
CATEGORY_NAMES = (
    'costs',
    'losses',
    'target_matrices',
    'realized_matrices',
    'reflection_matrices',
    'optimized_shapes',
    'optimizer_states',
    'solver_data',
    'plots/losses',
    'plots/target_matrices',
    'plots/realized_matrices',
    'plots/reflection_matrices',
    'plots/errors',
    'plots/optimized_shapes',
)


def layer_output_dir(layer_name: str) -> Path:
    return RUN_ROOT / layer_name


def runner_workspace(layer_name: str) -> Path:
    return layer_output_dir(layer_name) / '_runner_workspace'


def category_path(layer_name: str, category: str) -> Path:
    return layer_output_dir(layer_name) / category


def ensure_layer_categories(layer_name: str) -> dict[str, Path]:
    base = layer_output_dir(layer_name)
    base.mkdir(parents=True, exist_ok=True)
    paths = {name: base / name for name in CATEGORY_NAMES}
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)

    target_matrix = np.asarray(targets[layer_name], dtype=np.complex128)
    np.save(paths['target_matrices'] / f'{layer_name}_target_matrix.npy', target_matrix)
    shutil.copy2(TARGET_COPY_DIR / f'layer_{layer_name}.npy', paths['target_matrices'] / f'layer_{layer_name}.npy')
    return paths


def archive_path(src: Path, dest: Path, path_map: dict[str, str], *, move: bool = True) -> None:
    if not src.exists():
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        if dest.is_dir():
            shutil.rmtree(dest)
        else:
            dest.unlink()
    if move:
        shutil.move(str(src), str(dest))
    elif src.is_dir():
        shutil.copytree(src, dest)
    else:
        shutil.copy2(src, dest)
    path_map[str(src)] = str(dest)
    path_map[str(src.resolve())] = str(dest.resolve())


def replace_paths(value, path_map: dict[str, str]):
    if isinstance(value, str):
        return path_map.get(value, value)
    if isinstance(value, list):
        return [replace_paths(item, path_map) for item in value]
    if isinstance(value, dict):
        return {key: replace_paths(item, path_map) for key, item in value.items()}
    return value


def organize_runner_outputs(runner: MatrixInverseDesignRunner) -> dict:
    layer_name = runner.layer.layer_name
    paths = ensure_layer_categories(layer_name)
    workspace = runner.output_dir
    path_map = {}

    for src in workspace.glob('cost_estimate*.json'):
        archive_path(src, paths['costs'] / src.name, path_map)
    for src in workspace.glob('estimate_forward_batch*.hdf5'):
        archive_path(src, paths['costs'] / src.name, path_map)
    for src in workspace.glob('scratch_iteration*_loss_summary.json'):
        archive_path(src, paths['losses'] / src.name, path_map)
    for src in workspace.glob('current_matrix_iteration_*.npy'):
        if src.name.startswith('current_reflection_matrix_'):
            continue
        archive_path(src, paths['realized_matrices'] / src.name, path_map)
    for src in workspace.glob('current_reflection_matrix_iteration_*.npy'):
        archive_path(src, paths['reflection_matrices'] / src.name, path_map)
    for src in workspace.glob('density_iteration_*.npy'):
        archive_path(src, paths['optimized_shapes'] / src.name, path_map)
    for src in workspace.glob('eps_iteration_*.npy'):
        archive_path(src, paths['optimized_shapes'] / src.name, path_map)
    for pattern in ('optimizer_state_iteration_*.npz', 'adam_m_iteration_*.npy', 'adam_v_iteration_*.npy'):
        for src in workspace.glob(pattern):
            archive_path(src, paths['optimizer_states'] / src.name, path_map)
    for solver_dir_name in ('beamz_data', 'tidy3d_data'):
        solver_src = workspace / solver_dir_name
        if solver_src.exists():
            for src in solver_src.iterdir():
                archive_path(src, paths['solver_data'] / src.name, path_map)
            if solver_src.exists() and not any(solver_src.iterdir()):
                solver_src.rmdir()

    plot_rules = (
        ('scratch_iteration*_loss_terms*.png', 'plots/losses'),
        ('target_matrix_abs_iteration_*.png', 'plots/target_matrices'),
        ('target_vs_current_abs_iteration_*.png', 'plots/realized_matrices'),
        ('current_matrix_abs_iteration_*.png', 'plots/realized_matrices'),
        ('current_reflection_abs_iteration_*.png', 'plots/reflection_matrices'),
        ('current_matrix_abs_error_iteration_*.png', 'plots/errors'),
        ('*_optimized_eps_iteration_*.png', 'plots/optimized_shapes'),
    )
    for pattern, category in plot_rules:
        for src in workspace.glob(pattern):
            archive_path(src, paths[category] / src.name, path_map)

    summaries = sorted(paths['losses'].glob('scratch_iteration*_loss_summary.json'))
    for summary_path in summaries:
        summary = json.loads(summary_path.read_text())
        summary = replace_paths(summary, path_map)
        summary['notebook_output_layout'] = {
            'root': str(layer_output_dir(layer_name)),
            'category_folders': {name: str(path) for name, path in paths.items()},
            'runner_workspace': str(workspace),
            'path_rewrites': path_map,
        }
        summary_path.write_text(json.dumps(summary, indent=2, default=json_default) + '\n')

    if workspace.exists():
        for leftover in list(workspace.iterdir()):
            if leftover.is_dir() and not any(leftover.iterdir()):
                leftover.rmdir()
        if not any(workspace.iterdir()):
            workspace.rmdir()

    manifest = {
        'backend': BACKEND,
        'layer_name': layer_name,
        'target_matrix': str(paths['target_matrices'] / f'{layer_name}_target_matrix.npy'),
        'root': str(layer_output_dir(layer_name)),
        'category_folders': {name: str(path) for name, path in paths.items()},
        'latest_summary': str(summaries[-1]) if summaries else None,
    }
    manifest_path = layer_output_dir(layer_name) / 'artifact_manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2, default=json_default) + '\n')
    manifest['manifest_path'] = str(manifest_path)
    return manifest


def continuation_inputs(layer_name: str, start_iteration: int) -> dict:
    if int(start_iteration) <= 0:
        return {}
    return {
        'start_density': category_path(layer_name, 'optimized_shapes') / f'density_iteration_{int(start_iteration):02d}_after_update.npy',
        'optimizer_state': category_path(layer_name, 'optimizer_states') / f'optimizer_state_iteration_{int(start_iteration):02d}.npz',
        'existing_summary': category_path(layer_name, 'losses') / f'scratch_iteration{int(start_iteration)}_loss_summary.json',
    }


def expected_artifacts(layer_name: str, final_iteration: int) -> dict:
    paths = ensure_layer_categories(layer_name)
    stage = 'beamz_forward_pre_update'
    return {
        'target_matrix': str(paths['target_matrices'] / f'{layer_name}_target_matrix.npy'),
        'summary_json': str(paths['losses'] / f'scratch_iteration{int(final_iteration)}_loss_summary.json'),
        'cost_estimate_json': str(paths['costs'] / f'cost_estimate_scratch_iteration{int(final_iteration)}.json'),
        'realized_matrix_npy': str(paths['realized_matrices'] / f'current_matrix_iteration_{int(final_iteration):02d}_{stage}.npy'),
        'reflection_matrix_npy': str(paths['reflection_matrices'] / f'current_reflection_matrix_iteration_{int(final_iteration):02d}_{stage}.npy'),
        'density_npy': str(paths['optimized_shapes'] / f'density_iteration_{int(final_iteration):02d}_after_update.npy'),
        'eps_npy': str(paths['optimized_shapes'] / f'eps_iteration_{int(final_iteration):02d}_after_update.npy'),
        'optimizer_state_npz': str(paths['optimizer_states'] / f'optimizer_state_iteration_{int(final_iteration):02d}.npz'),
        'solver_iteration_dir': str(paths['solver_data'] / f'{layer_name}_opt_iter_{int(final_iteration):02d}'),
    }


def write_run_manifest(runners: dict[str, MatrixInverseDesignRunner], *, note: str) -> Path:
    manifest_dir = RUN_ROOT / 'manifests'
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest = {
        'backend': BACKEND,
        'note': note,
        'target_manifest': str(TARGET_COPY_DIR / 'target_manifest.json'),
        'run_root': str(RUN_ROOT),
        'layout': 'backend/layer/category; continuation uses the same stable layer folder',
        'category_names': list(CATEGORY_NAMES),
        'layers': {
            name: expected_artifacts(name, int(runner.config.iterations))
            for name, runner in runners.items()
        },
    }
    path = manifest_dir / 'notebook_artifact_manifest.json'
    path.write_text(json.dumps(manifest, indent=2, default=json_default) + '\n')
    return path


## Configure BeamZ Runners

Runtime outputs are archived under `example/outputs/beamz_3layer_routing5_strict/<layer>/` by category, not by iteration span.


In [ ]:
BACKEND = 'beamz'
RUN_ROOT = OUTPUT_BASE / 'beamz_3layer_routing5_strict'

ITERATIONS = 5
START_ITERATION = 0
LEARNING_RATE = 0.05
FILTER_RADIUS_UM = 0.15
PROJECTION_BETA = 50.0
RUN_TIME_PS = 15.0
REFLECTION_PENALTY_WEIGHT = 1.0
MAX_INPUTS = 16


def make_runner(layer_name: str, **overrides) -> MatrixInverseDesignRunner:
    iterations = int(overrides.pop('iterations', ITERATIONS))
    start_iteration = int(overrides.pop('start_iteration', START_ITERATION))
    resume_kwargs = continuation_inputs(layer_name, start_iteration)
    resume_kwargs.update(overrides)
    config = MatrixInverseDesignRunConfig(
        targets=TARGETS_PATH,
        output_dir=runner_workspace(layer_name),
        backend=BACKEND,
        layer_name=layer_name,
        max_inputs=MAX_INPUTS,
        start_iteration=start_iteration,
        iterations=iterations,
        learning_rate=LEARNING_RATE,
        filter_radius_um=FILTER_RADIUS_UM,
        projection_beta=PROJECTION_BETA,
        run_time_ps=RUN_TIME_PS,
        reflection_penalty_weight=REFLECTION_PENALTY_WEIGHT,
        local_gradient=True,
        max_num_adjoint_per_fwd=1,
        task_name_prefix=f'lumix_beamz_r5_{layer_name}_iter{iterations}',
        overwrite=True,
        **resume_kwargs,
    )
    return MatrixInverseDesignRunner(config)

runners = {name: make_runner(name) for name in LAYER_NAMES}
manifest_path = write_run_manifest(runners, note='Expected BeamZ notebook artifacts before execution.')
[(name, runner.backend.name, runner.output_dir, len(runner.layer.task_names)) for name, runner in runners.items()], manifest_path


## Estimate Local BeamZ Workload

This is local only and consumes no FlexCredits.


In [ ]:
ESTIMATE_BEAMZ_WORKLOAD = True

if ESTIMATE_BEAMZ_WORKLOAD:
    workload_estimates = {}
    manifests = {}
    for name in LAYER_NAMES:
        runner = make_runner(name)
        workload_estimates[name] = runner.estimate_cost()
        manifests[name] = organize_runner_outputs(runner)
    manifest_path = write_run_manifest(runners, note='BeamZ local workload estimates archived by category; no FlexCredits used.')
    workload_estimates, manifests, manifest_path
else:
    print('Set ESTIMATE_BEAMZ_WORKLOAD = True to write local workload estimates.')


## Plot BeamZ Simulation Domain

This uses BeamZ native `Simulation.plot_eps()` on the selected layer and saves the initial density/eps arrays.


In [ ]:
PLOT_LAYER = 'subunitary_0'

runner = make_runner(PLOT_LAYER)
paths = ensure_layer_categories(PLOT_LAYER)
density, _, _ = runner.load_start_state()
geometry = runner.backend.build_geometry(runner.layer, density=density, config=runner.design_config)
eps = runner.backend.density_to_beamz_eps(runner.design_config, density)
np.save(paths['optimized_shapes'] / f'{PLOT_LAYER}_beamz_initial_density.npy', density)
np.save(paths['optimized_shapes'] / f'{PLOT_LAYER}_beamz_initial_eps.npy', eps)

print('resolution_m:', geometry.resolution_m)
print('domain_bounds_m:', geometry.domain_bounds_m)
print('grid_spec:', geometry.simulation.grid_spec)

geometry.simulation.plot_eps()
fig = plt.gcf()
fig.set_size_inches(12, 5)
plt.title(f'BeamZ eps domain: {PLOT_LAYER}')
plot_path = paths['plots/optimized_shapes'] / f'{PLOT_LAYER}_beamz_native_eps.png'
fig.savefig(plot_path, dpi=180, bbox_inches='tight')
plt.show()

{'beamz_eps_plot': str(plot_path), 'initial_density': str(paths['optimized_shapes'] / f'{PLOT_LAYER}_beamz_initial_density.npy'), 'initial_eps': str(paths['optimized_shapes'] / f'{PLOT_LAYER}_beamz_initial_eps.npy')}


## Run BeamZ Three-Layer Optimization

The call shape matches the Tidy3D notebook. BeamZ uses local forward modal DFT evaluations and native adjoint overlap-gradient updates, then maps gradients through a JAX VJP of the same Lumix/Tidy3D filter and smoothed projection math. This consumes no FlexCredits but can still be computationally heavy for the full 16-port, 3-layer run.


In [ ]:
RUN_BEAMZ_OPTIMIZATION = False

if RUN_BEAMZ_OPTIMIZATION:
    results = {}
    manifests = {}
    for name in LAYER_NAMES:
        runner = make_runner(name)
        results[name] = runner.run(allow_cloud_run=True)
        manifests[name] = organize_runner_outputs(runner)
        print(name, manifests[name]['latest_summary'])
    manifest_path = write_run_manifest(runners, note='BeamZ optimization archived by category.')
    results, manifests, manifest_path
else:
    print('Set RUN_BEAMZ_OPTIMIZATION = True to run the local BeamZ optimizer.')
